# 13. TensorFlow Mini-Project: Medical Diagnostic Imaging Classifier

### Capstone Objective:
Build, train, evaluate, and persist an end-to-end Deep Convolutional Neural Network for automated medical diagnostic image classification (e.g., Pneumonia vs Normal Chest X-Ray detection) using TensorFlow 2.x and Keras.

### Pipeline Stages:
1. Synthetic Medical Imaging Generation & Data Normalization
2. Deep CNN Architecture with BatchNormalization & Spatial Dropout
3. Learning Rate Scheduling & EarlyStopping Callbacks
4. Confusion Matrix, ROC-AUC, and F1-Score Evaluation
5. Production Model Persistence in Native `.keras` Format

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import numpy as np
from sklearn.metrics import classification_report, roc_auc_score

# 1. Synthesize Medical Chest X-Ray Imaging Data (600 samples, 32x32 grayscale)
np.random.seed(42)
X_imgs = np.random.uniform(0, 255, (600, 32, 32, 1)).astype(np.float32) / 255.0
# Synthetic pathology: central pixel intensity correlates with positive pneumonia
y_labels = (X_imgs[:, 12:20, 12:20, 0].mean(axis=(1,2)) > 0.52).astype(np.int32)

# Train/Test partition (80/20)
X_tr, X_te = X_imgs[:480], X_imgs[480:]
y_tr, y_te = y_labels[:480], y_labels[480:]

# 2. Build CNN Architecture
model = models.Sequential([
    layers.Input(shape=(32, 32, 1)),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 3. Train with EarlyStopping
es = callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
history = model.fit(X_tr, y_tr, epochs=8, batch_size=32, validation_split=0.2, callbacks=[es], verbose=1)

# 4. Evaluation
test_loss, test_acc, test_auc = model.evaluate(X_te, y_te, verbose=0)
print(f"\nHoldout Diagnostic Test Accuracy: {test_acc*100:.2f}%")
print(f"Holdout Diagnostic Test ROC-AUC:  {test_auc:.4f}")

## 🎯 Exercises & Practice
1. Save the trained medical diagnostic pipeline to disk and perform a mock single-patient inference test.

In [ ]:
# Exercise Solution
import os, tempfile
model_path = os.path.join(tempfile.gettempdir(), 'pneumonia_detector.keras')
model.save(model_path)

# Single patient inference
patient_scan = X_te[0:1]
risk_score = float(model.predict(patient_scan, verbose=0)[0][0])
print(f"Patient Risk Score: {risk_score:.4f} -> Diagnostic: {'PNEUMONIA DETECTED' if risk_score > 0.5 else 'NORMAL SCAN'}")